# Compiled vectorizer bottleneck experiment

Profile full vectorization on a few cached protein graphs. Then compare the nested simplex-pair index loop in Python and Numba, and compare the complete feature-count path using Python's standard JSON encoder versus `orjson`'s native Rust encoder. Both comparisons check exact outputs.

Install the optional experiment dependencies with `python -m pip install numba orjson`. The Numba timing excludes its first-call compilation time.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'notebooks' / 'src' / 'enzyme_workflow').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root from this notebook's working directory")
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks' / 'src'))

from enzyme_workflow.common import add_project_sources
PROJECT_ROOT = add_project_sources(PROJECT_ROOT)
from enzyme_workflow.compiled_bottleneck_experiment import (
    benchmark_compiled_pair_loop,
    benchmark_native_json,
)

In [ ]:
SAMPLE_COUNT = 3
MAX_SIMPLEX_DIMENSION = 3
MAX_REPRESENTATIVE_DISTANCE = 1
REPEATS = 3

In [ ]:
pair_loop_results, pair_loop_summary = benchmark_compiled_pair_loop(
    PROJECT_ROOT,
    sample_count=SAMPLE_COUNT,
    max_simplex_dimension=MAX_SIMPLEX_DIMENSION,
    max_distance=MAX_REPRESENTATIVE_DISTANCE,
    repeats=REPEATS,
)
display(pair_loop_summary)
display(pair_loop_results)

In [ ]:
json_results, json_summary = benchmark_native_json(
    PROJECT_ROOT,
    sample_count=SAMPLE_COUNT,
    repeats=REPEATS,
)
display(json_summary)
display(json_results)

## Interpretation

The compiled pair-loop table isolates index generation; it does not compile the subsequent JSON serialization, BLAKE2 hashing, or counter updates. The profile estimates the time in those remaining stages. The `orjson` comparison runs the full per-graph feature-count operation and verifies that the hashed outputs are identical. Neither experiment changes the production vectorizer.